### Questions of Interest
* The bucket sizes Simmons uses were an arbitrary decision that was derived from the concept of reducing variation to reduce waste. How many buckets should Simmons be using and what sizes should the buckets be?
* Given Simmons target to eventually hit 75% upgrade, what "acceptable trim percentage" should be considered for determining bucket sizes?
* Simmons has expressed that they are hesitant to increase the probability of miscuts because they do not currently have the capacity to handle large amounts. Is there an acceptable tradeoff between trim and miscuts that will minimize yield? Is this path worth exploring?

# How do we account for the variation of weights in bucket size for calculating waste?

This dataset outlines three days worth of measurements from the x-ray machines on the processing line from Simmons poultry processing facility in Gentry, AR. The machines measure the weight (in pounds) of individual chicken breasts before they are sent to portioning facilities. The dataset contains three distributions: trim (small offcuts that mistakenly end up on the breast conveyors), single breasts, and butterflies (both breasts removed from the bird in one piece).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

from scipy.stats import norm
from scipy.signal import argrelextrema
from scipy.stats import skewnorm
from scipy.stats import chi2
from scipy.optimize import minimize

In [ ]:
# ingest data
df = pd.DataFrame(pd.read_csv('data/piecewise_data1.csv'))
df[['Weight (g)']] = df[['Weight (lb)']] * 453.592 # convert lbs to grams
x = df[['Weight (g)']].values
df.head()

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(x, color='blue')

plt.xlabel("Weight (g)")
plt.ylabel("Density")
plt.show()

## Identify Minima of Intrest
We are interested in the weights of single chicken breasts. The first step in preprocessing this dataset is to identify the bounds of my distribution of interest. To do that I need to find the minima surrounding this distribution.

In [ ]:
plt.figure(figsize=(12, 6))
kde_plot = sns.kdeplot(x, bw_adjust=1, fill=False, linewidth=2, color='black')
xs, ys = kde_plot.get_lines()[0].get_data()

# Find local minima (valleys)
minima_idx = argrelextrema(ys, np.less)[0]
split_points = xs[minima_idx[:2]]
left, right = split_points[0], split_points[1]

# Plot histogram + KDE + split points
plt.clf()  # clear previous figure

plt.figure(figsize=(12, 6))
sns.histplot(x, kde=True, color='skyblue', alpha=0.5)

# Add split point lines
for sp in split_points:
    plt.axvline(sp, color='red', linestyle='--', linewidth=2)

plt.title(f"Split points at density minima: {split_points}")
plt.xlabel("Weight (g)")
plt.ylabel("Density")
plt.show()

## Fit Normal Distribution
After finding the minima of the distribution, I can trim the unnecessary data and begin fitting distributions to the dataset.

In [ ]:
# Trim data between upper and lower bound
trimmed = x[(x >= left) & (x <= right)]
x_train, x_test = train_test_split(trimmed, test_size=0.3, random_state=42)

In [ ]:
# Fit a normal distribution
mu, sigma = norm.fit(x_train)

# Plot trimmed data + fitted normal curve
plt.figure(figsize=(12, 6))

# Histogram of trimmed data
sns.histplot(x_train, bins=30, stat='density', color='skyblue', alpha=0.5)

# Normal curve
x_fit = np.linspace(x_train.min(), x_train.max(), 500)
normPDF = norm.pdf(x_fit, mu, sigma)
plt.plot(x_fit, normPDF, 'r-', linewidth=2, label='Fitted Normal')

plt.title(f"Fitted Normal: mean = {mu:.3f}, sigma = {sigma:.3f}")
plt.xlabel("Weight (g)")
plt.ylabel("Density")
plt.legend()
plt.show()

## Fit Skewed Normal Distribution
After fitting the normal distribution, the histogram looked to me like it had a slight right skew to it. Taking the same data and using the same package, I fit the skewed normal distribution.

In [ ]:
# Fit skew-normal distribution
a, loc, scale = skewnorm.fit(x_train)

# Plot histogram + fitted skew-normal curve
plt.figure(figsize=(12, 6))

# Histogram of trimmed data
sns.histplot(x_train, bins=30, stat='density', color='skyblue', alpha=0.5)

# Skew-normal PDF
x_fit = np.linspace(x_train.min(), x_train.max(), 500)
skewnormPDF = skewnorm.pdf(x_fit, a, loc, scale)
plt.plot(x_fit, skewnormPDF, 'r-', linewidth=2, label='Fitted Skew-Normal')

plt.title(f"Skewnorm parameters: a={a:.3f}, loc={loc:.3f}, scale={scale:.3f}")
plt.xlabel("Weight (g)")
plt.ylabel("Density")
plt.legend()
plt.show()

## Compare Fits
To compare the fits of the two distributions I calculated the MSE and found that the difference between the two was negligible. For simplicity's sake, I will continue using the normal distribution for future analysis.

In [ ]:
def chi_squared_gof(data_column, distribution):
    data = np.asarray(data_column)
    n = len(data)

    # Fit distribution
    params = distribution.fit(data)

    # Number of bins (Freedman–Diaconis rule)
    q75, q25 = np.percentile(data, [75, 25])
    iqr = q75 - q25
    bin_width = 2 * iqr * n ** (-1/3)
    bins = int(np.ceil((data.max() - data.min()) / bin_width))

    # Observed frequencies
    observed, bin_edges = np.histogram(data, bins=bins)

    # Expected frequencies
    cdf_vals = distribution.cdf(bin_edges, *params)
    expected = np.diff(cdf_vals) * n

    # Remove bins where expected == 0
    mask = expected > 0
    observed = observed[mask]
    expected = expected[mask]

    # Chi-squared statistic
    chi_stat = np.sum((observed - expected) ** 2 / expected)

    # Degrees of freedom:
    # (# bins - 1) - (# estimated parameters)
    dof = len(observed) - 1 - len(params)

    p_value = 1 - chi2.cdf(chi_stat, dof)

    return p_value, dof

norm_chi2, norm_dof = chi_squared_gof(x_test, norm)
skewnorm_chi2 = chi_squared_gof(x_test, skewnorm)
print("Chi-Squared P-value of normal distribution: " + str(norm_chi2), norm_dof)
print("Chi-Squared P-value of skewnorm distribution: " + str(skewnorm_chi2))

# What is the average unavoidable trim for each bucket size?
Simmons would like us to target the lower bound of the weight bucket when making portioning decisions in order to minimize miscuts. When targeting the lower bound, there is a proportion of meat in each breast that is inevitably lost from breasts any larger than the absolute minimum. I want to explore what exactly that amount of waste is to see if modified bucket sizes or a different target weight rule could make a significant impact on the total waste of any given portioning decision.

### Expected Value of a Truncated Normal Distribution

$$ E[X] = \mu + \sigma\frac{\phi(\frac{a-\mu}{\sigma}) - \phi(\frac{b-\mu}{\sigma})}{\Phi(\frac{b-\mu}{\sigma}) - \Phi(\frac{a-\mu}{\sigma})}$$

Where
 * $\phi =$ The standard normal PDF
 * $\Phi =$ The standard normal CDF
 * $a,b = lb,ub$ of bucket range


In [ ]:
def expected_x(a, b, mu, sigma):
    alpha = (a - mu)/sigma
    beta = (b - mu)/sigma
    mean = mu + sigma*((norm.pdf(alpha)-norm.pdf(beta))/(norm.cdf(beta)-norm.cdf(alpha)))
    return (mean - a) / mean

print(expected_x(440, 490, mu, sigma))

### Estimating Breast Weight using Monte Carlo Simulation
My next step was to define a function that would preform a Monte Carlo estimate of $E[f(X) | a <= X <= b]$, where:
 * $f(X)$ ~ $Normal(\mu, \sigma^2)$
 * $a,b = lb,ub$ of bucket range
 * $U = UniformRV(f(a), f(b))$
 * $X = f^{-1}(U)$

Returns:
   * $\hat{\mu} =$ Estimated average of bucket range
   * $h = $ Half-width of estimate
   * $E[x] = $ Expected trim as a percentage of average weight

This function is a 3-step process. First, it finds the P-values of $a$ and $b$ and generates $n$ random variates from $U$. Then it enters all $n$ variates into the inverse CDF of the normal distribution to return their weight values in grams. Finally, it calculates and returns the desired statistics.

In [ ]:
def mc_truncated_avg_pdf(mu, sigma, a, b, alpha=0.05):
    # Finds the P-value of the buckets upper and lower bounds
    Fa = norm.cdf(a, loc=mu, scale=sigma)
    Fb = norm.cdf(b, loc=mu, scale=sigma)

    # Generates an RV from the normal distribution within a given bucket
    U = np.random.uniform(Fa, Fb, n)
    X = norm.ppf(U, loc=mu, scale=sigma)

    # Calculates mean and HW with desired confidence
    mean_est = X.mean()
    s = X.std()
    z = norm.ppf(1 - alpha / 2)
    ci_half = z * s / np.sqrt(n)

    # Calculates the expected unavoidable trim
    expected_trim = (mean_est - a) / mean_est

    return mean_est, ci_half, expected_trim

### Determine Required Sample Size
To preform any sort of monte carlo simulation, I must first determine the required sample size to achieve my desired confidence. For this situation I am interested in finding the mean weight (g) +/- 1g with 95% confidence. Using normal approximation, the required sample size can be found using the equation:

$$n≥(\frac{z_{1−(α/2)}s}{ϵ})^2$$

Where ϵ is the desired half-width. Since the sample standard deviation I got from the normal fit is larger than the standard deviation from any of the bucket sizes, this number will be inflated to some extent. Regardless, it will provide me with at least my desired confidence.

In [ ]:
n = pow((1.96*sigma/1.0), 2).astype(int)
n

Simmons uses eight weight buckets, ranging from 0g to 1000g. I add all these buckets to a dictionary and iterate through the list while applying the Monte Carlo simulation method from above. This collects the estimated average weight of a breast, half-width of the estimate, and expected trim percentage associated with the bucket. and outputs the results to the console.

In [ ]:
# Creates bucket ranges dictionary
ranges = {
#    "0-390": (0, 390), # Using this bucket with the lb rule results in a trim% of 1
    "390-440": (390, 440),
    "440-490": (440, 490),
    "490-540": (490, 540),
    "540-590": (540, 590),
    "590-640": (590, 640),
    "640-690": (640, 690),
    "690-1000": (690, 1000)
}

# Simulates buckets and adds results to dict
results = {
    label: dict(
        zip(
            ["estimated_mean", "95%_CI_halfwidth", "expected_trim%"],
            mc_truncated_avg_pdf(mu, sigma, low, high),
 #           mc_truncated_avg_pdf(502, 90.63, low, high) # CV model that Simmons provided
        )
    )
    for label, (low, high) in ranges.items()
}

# Print results
print(json.dumps(results, indent=2))


# What is the Impact of Increasing the Number of Buckets?
Increasing the number of buckets that Simmons uses would decrease the difference between the lowest and highest weight in the range. This would in turn reduce the average trim percentage in every bucket but would come at the cost of potentially increasing space requirements and ordering complexity for the portioning plants.

In [ ]:
base_low = 390
base_high = 690

partition_counts = [6, 7, 8, 9, 10, 11, 12]

def build_subranges(a, b, n_parts):
    edges = np.linspace(a, b, n_parts + 1)
    return {
        f"{round(edges[i],3)}-{round(edges[i+1],3)}": (edges[i], edges[i+1])
        for i in range(n_parts)
    }

results = {
    n_parts: {
            label: dict(
                zip(
                    ["estimated_mean", "95%_CI_halfwidth", "expected_trim%"],
                    mc_truncated_avg_pdf(502, 90.63, low, high),
                )
            )
        for label, (low, high) in build_subranges(base_low, base_high, n_parts).items()
    }
    for n_parts in partition_counts
}

avg_trim_by_partition = {
    n_parts: np.mean([bucket["expected_trim%"]
                      for bucket in part_results.values()])
    for n_parts, part_results in results.items()
}

print(json.dumps(avg_trim_by_partition, indent=2))

# What Bucket Sizes Minimize the Difference in Trim Percentage?
Iterators:

  * $K = 6$ number of buckets

Parameters:

  * $\mu,\sigma =$ parameters of the normal distribution
  * $L,U = lb,ub$ of normal distribution

Variables:

  * $a_k,b_k = lb,ub$ of bucket range k

Objective: Minimize the total variation in the proportion of excess over the lower bound

  * $\alpha_k = \frac{a_k-\mu}{\sigma}$ standardized lower bound
  * $\beta_k = \frac{b_k-\mu}{\sigma}$ standardized upper bound
  * $f(a_k,b_k) = \mu + \sigma\frac{\phi(\alpha_k) - \phi(\beta_k)}{\Phi(\beta_k) - \Phi(\alpha_k)}$ expected value of truncated normal distribution

min $$\sum_{k = 1}^K \frac{f(a_k, b_k)-a_k}{f(a_k, b_k)}$$

Subject to:

  * $a_k = b_{k-1} \forall$ k in 2..K
  * $a_1 = L$
  * $b_K = U$


In [ ]:
# TODO talk to Dr. Sullivan about what optimization method would best fit this model

# Does Accepting Miscuts Reduce Trim?
Simmons does not currently have the capacity to handle a large number of miscuts. This question explores if it is worth accepting some miscuts as a tradeoff for reducing the total amount of waste. To find the answer to this question, I modified the Monte Carlo simulation to account for

if

$w$ = weight (g) of a randomly selected breast from bucket k

$u$ = weight (g) of portioning decision $\in$ bucket k

$\Theta(w, u)$ = the function for upgrade percentage given
$\Theta(w) = { \frac{u}{w}$ if $w >= u, 0 $otherwise$}$

$$ $$

In [ ]:
def miscut_simulation(max_miscut_percentage, bucket_lb, bucket_ub, mean, stdev) :
# f(w) = PDF of the truncated normal distribution from lb to ub given w
# function of upgrade percentage that returns 0 if w < u and (u/w) if w >= u
# upgrade: the integral from u to ub for u/w * f(w) with respect to w
# trim: 1 - upgrade
# return trim, upgrade
